In [0]:
from datetime import datetime
from pyspark.sql.functions import current_timestamp

# =========================
# PATHS
# =========================

bucket          = "retail-etl-dwh-lakehouse"

processed_path  = f"s3://{bucket}/processed/"
sftp_out_path   = f"s3://{bucket}/sftp_out/"

today = datetime.now().strftime("%d%m%Y")

# =========================
# SET CORRECT CATALOG + SCHEMA
# =========================

catalog_name = "`retail-dwh-project`"
schema_name  = "gold"     # change if your schema is warehouse/bronze/etc

# =========================
# TABLES
# =========================

tables = [
    "dim_customer",
    "dim_product",
    "dim_store",
    "fact_sales"
]

print(f"\n=== Export Started ===")
print(f"Catalog : {catalog_name}")
print(f"Schema  : {schema_name}")
print(f"Date    : {today}")

# =========================
# EXPORT LOOP
# =========================

for table in tables:

    full_table = f"{catalog_name}.{schema_name}.{table}"

    print(f"\nProcessing Table: {full_table}")

    try:

        # Read table
        df = spark.table(full_table) \
                  .withColumn("ExportedAt", current_timestamp())

        # Temp + final paths
        temp_path  = f"{processed_path}{today}/{table}_tmp/"
        final_file = f"{sftp_out_path}{today}/{table}_{today}.csv"

        # Write single CSV
        df.coalesce(1) \
          .write.mode("overwrite") \
          .option("header", True) \
          .csv(temp_path)

        # Rename part file
        for f in dbutils.fs.ls(temp_path):

            if f.name.startswith("part-"):

                dbutils.fs.cp(f.path, final_file)

                print(f"  ✅ Exported: {final_file}")

        # Remove temp folder
        dbutils.fs.rm(temp_path, recurse=True)

        print(f"  ✅ Rows Exported: {df.count()}")

    except Exception as e:

        print(f"  ❌ ERROR in {table}")
        print(e)

# =========================
# FINAL OUTPUT
# =========================

print(f"\n=== Files in sftp_out/{today}/ ===")

try:

    for f in dbutils.fs.ls(f"{sftp_out_path}{today}/"):

        if f.name.endswith(".csv"):

            print(f"  ✅ {f.name}")

except Exception as e:

    print("No files exported yet")
    print(e)

print("\n=== Export Completed ===")